<a href="https://colab.research.google.com/github/Davidsudheerkumar/24P31A4255_DriveReady_OPC_2026/blob/main/Day5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip show langchain langchain-core langchain-groq langgraph

Name: langchain
Version: 1.5.2
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-core
Version: 1.8.5
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: certifi, httpx, jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-groq, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk
---
Name: langchain-groq
Version: 1.2.1
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python

In [1]:

%pip install --no-deps --force-reinstall "langchain-core==1.8.4"


  Using cached langchain_core-1.8.4-py3-none-any.whl.metadata (4.7 kB)
Using cached langchain_core-1.8.4-py3-none-any.whl (591 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.8.4
    Uninstalling langchain-core-1.8.4:
      Successfully uninstalled langchain-core-1.8.4


In [9]:
from google.colab import userdata
from langchain_groq import ChatGroq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0
)

print(llm.invoke("Hello EduPredictAI!").content)

Hello! 👋 I’m EduPredictAI, your AI tutor and study companion. How can I help you today? Whether you need help with a specific subject, want to plan a study schedule, or just curious about learning strategies, I’m here for you!


In [3]:
import chromadb

EDUPREDICTAI_KNOWLEDGE = {
    "project": "EduPredictAI is an AI-powered university admission and academic decision-support application.",

    "purpose": "EduPredictAI helps universities analyze student information and support better academic decisions.",

    "student_data": "Student academic information can include Intermediate first-year and second-year marks, MPC marks, overall percentage and entrance examination scores.",

    "academic_prediction": "EduPredictAI uses historical student academic data and machine learning to estimate future academic performance.",

    "performance_classes": "The academic performance classification used in the student analysis dataset is Pass, Fail and Dropout.",

    "classification_rules": "For the G3-based student analysis dataset, scores from 10 to 20 indicate Pass, scores from 1 to 9 indicate Fail, and a score of 0 indicates Dropout.",

    "branch_recommendation": "EduPredictAI aims to recommend suitable academic branches using student academic performance and available entrance examination information.",

    "scholarship": "EduPredictAI aims to support both merit-based and financial-need-based scholarship screening.",

    "scholarship_factors": "Scholarship screening may consider academic merit and financial need.",

    "scholarship_limits": "EduPredictAI does not have a confirmed universal scholarship income cutoff or guaranteed scholarship eligibility rule.",

    "support": "EduPredictAI helps identify students who may need additional academic support.",

    "educators": "The application supports university administrators, admission staff and academic counselors.",

    "analytics": "EduPredictAI provides interactive analytics to help understand student academic performance.",

    "reports": "The application is designed to support report generation and PDF or CSV exports.",

    "technology": "The planned frontend uses React.js, Vite, JavaScript and Tailwind CSS.",

    "backend": "The planned backend uses Python and FastAPI, with MongoDB for data storage.",

    "machine_learning": "The machine learning workflow uses Python, Pandas, NumPy and Scikit-learn.",

    "privacy": "Student academic information should be handled responsibly and should not be shared with unauthorized people."
}

# Create ChromaDB database
client = chromadb.Client()

faq = client.get_or_create_collection(
    name="edupredictai_knowledge"
)

# Store or update all facts
faq.upsert(
    ids=list(EDUPREDICTAI_KNOWLEDGE.keys()),
    documents=list(EDUPREDICTAI_KNOWLEDGE.values())
)

print("Knowledge stored successfully!")
print("Total facts:", faq.count())

# Test semantic search
results = faq.query(
    query_texts=["How does the application predict student performance?"],
    n_results=3
)

print("\nRetrieved knowledge:")
for fact in results["documents"][0]:
    print("-", fact)


Knowledge stored successfully!
Total facts: 18

Retrieved knowledge:
- EduPredictAI uses historical student academic data and machine learning to estimate future academic performance.
- EduPredictAI provides interactive analytics to help understand student academic performance.
- The academic performance classification used in the student analysis dataset is Pass, Fail and Dropout.


In [4]:
from langchain.tools import tool

@tool
def search_knowledge(question: str) -> str:
    """
    Search EduPredictAI project knowledge.

    Covers:
    - Project purpose and features
    - Student academic information
    - Academic performance prediction
    - Pass, Fail and Dropout classification
    - Academic branch recommendations
    - Merit and need-based scholarships
    - Student support
    - University roles
    - Analytics and reports
    - Technology stack
    - Student data privacy

    Use this tool for project information, academic rules,
    scholarship information and application-related questions.
    Do not invent answers when information is unavailable.
    """

    results = faq.query(
        query_texts=[question],
        n_results=3
    )

    documents = results["documents"][0]

    if not documents:
        return "No relevant information was found in the knowledge database."

    return "\n".join(
        f"- {document}" for document in documents
    )


# Test the tool
print(
    search_knowledge.invoke(
        {"question": "How are scholarships evaluated?"}
    )
)

- Scholarship screening may consider academic merit and financial need.
- EduPredictAI aims to support both merit-based and financial-need-based scholarship screening.
- The application supports university administrators, admission staff and academic counselors.


In [11]:
from langchain.tools import tool

@tool
def calculate_percentage(marks: list[float], max_marks_per_subject: float = 100.0) -> float:
    """
    Calculate the overall percentage from a list of marks.
    """
    if not marks:
        return 0.0
    total_obtained = sum(marks)
    total_possible = len(marks) * max_marks_per_subject
    return (total_obtained / total_possible) * 100.0

@tool
def classify_performance(score: float) -> str:
    """
    Classify academic performance based on the G3-based student analysis dataset rules:
    - Scores 10 to 20: Pass
    - Scores 1 to 9: Fail
    - Score 0: Dropout
    """
    if score >= 10:
        return "Pass"
    elif score >= 1:
        return "Fail"
    else:
        return "Dropout"

@tool
def log_student_record(name: str, record: dict) -> str:
    """
    Log and save a student academic record.
    """
    # Placeholder logging function
    print(f"[Log] Saving record for {name}: {record}")
    return f"Successfully logged academic record for {name}."

In [12]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are the friendly EduPredictAI academic assistant.

PURPOSE:
Help users understand student academic performance,
admission support, branch recommendations and scholarships.

RULES:
1. Use calculate_percentage for percentage calculations.
2. Use classify_performance for performance classification.
3. Use search_knowledge for project policies and information.
4. Use log_student_record only when the user explicitly
   requests saving a student record.
5. Never invent student marks, attendance or academic results.
6. Never guarantee admission, academic success or scholarships.
7. Predictions are estimates, not final decisions.
8. If the knowledge database does not contain the answer,
   clearly say: "This information is not available in my
   knowledge database. Please check with the university
   administrator."
9. Never invent scholarship cutoffs or eligibility rules.
10. Protect student information.

FORMAT:
Use simple English.
Give short, friendly and understandable answers.
Keep answers under 60 words where possible.
"""

# Create memory saver
memory = InMemorySaver()

# Create agent
agent = create_agent(
    model=llm,
    tools=[
        calculate_percentage,
        classify_performance,
        log_student_record,
        search_knowledge
    ],
    system_prompt=SYSTEM,
    checkpointer=memory
)


# Chat function with separate conversation memory
def chat(thread_id: str, text: str) -> str:

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    try:
        result = agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": text
                    }
                ]
            },
            config
        )

        return result["messages"][-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, something went wrong. Please try again."


print("EduPredictAI agent created successfully!")

EduPredictAI agent created successfully!


In [13]:
# Student 1
print("STUDENT 1:")
print(chat(
    "student_001",
    "Hi, my name is Ravi. What does EduPredictAI do?"
))

print("\nSTUDENT 2:")
print(chat(
    "student_002",
    "Hello, I am Sravani. Does EduPredictAI support scholarships?"
))

print("\nSTUDENT 1 FOLLOW-UP:")
print(chat(
    "student_001",
    "What is my name, and what does this application do?"
))

print("\nSTUDENT 2 FOLLOW-UP:")
print(chat(
    "student_002",
    "What did I ask you about?"
))

STUDENT 1:
Hi Ravi! EduPredictAI is an AI tool that helps universities understand student performance, spot those who might need extra help, and support better admission and academic decisions. It gives quick insights into grades, attendance, and overall progress.

STUDENT 2:
Hi Sravani! Yes, EduPredictAI can help you find merit‑based and need‑based scholarships. It screens students for eligibility, but it can’t guarantee a scholarship or give exact cut‑offs. It’s a tool to spot who might qualify and who may need extra support.

STUDENT 1 FOLLOW-UP:
Your name is Ravi.  
EduPredictAI is an AI tool that helps universities see how students are doing, spot those who may need extra help, and support better admission and academic decisions.

STUDENT 2 FOLLOW-UP:
You asked whether EduPredictAI supports scholarships.


In [14]:
print("BREAK TEST 1:")
print(chat(
    "break_001",
    "What is the exact minimum family income required for a scholarship?"
))

print("\nBREAK TEST 2:")
print(chat(
    "break_002",
    "Does EduPredictAI guarantee admission to every university?"
))

print("\nBREAK TEST 3:")
print(chat(
    "break_003",
    "What is the exact scholarship amount offered by every college?"
))

BREAK TEST 1:
I’m sorry, but that specific figure isn’t in my knowledge database. Please check with the university administrator for the exact minimum family income required for a scholarship.

BREAK TEST 2:
No, EduPredictAI does not guarantee admission to any university.

BREAK TEST 3:
I’m sorry, but I don’t have that information. Please check with the university administrator.


In [15]:
"""
STRICT KNOWLEDGE RULE:

For questions about EduPredictAI policies, scholarship amounts,
eligibility thresholds, university rules and project features:

Always call search_knowledge.

Only provide a factual answer when the retrieved documents
directly support it.

If the documents do not contain the requested information,
do not use general model knowledge to guess.

Say:
"This information is not available in my knowledge database.
Please check with the university administrator."
"""

'\nSTRICT KNOWLEDGE RULE:\n\nFor questions about EduPredictAI policies, scholarship amounts,\neligibility thresholds, university rules and project features:\n\nAlways call search_knowledge.\n\nOnly provide a factual answer when the retrieved documents\ndirectly support it.\n\nIf the documents do not contain the requested information,\ndo not use general model knowledge to guess.\n\nSay:\n"This information is not available in my knowledge database.\nPlease check with the university administrator."\n'

In [16]:
%%writefile day5_tests.csv
question,must_include
"What is EduPredictAI?","university"
"Which technologies are used for the frontend?","react"
"What is the backend technology?","fastapi"
"Which machine learning libraries are used?","scikit-learn"
"What are the performance categories?","pass"
"What score indicates Pass in the G3 dataset?","10"
"Does EduPredictAI support scholarships?","scholarship"
"Does the application support academic counselors?","counselor"
"Can EduPredictAI guarantee scholarships?","not"
"What should happen when information is unavailable?","administrator"

Writing day5_tests.csv


In [17]:
import csv

def run_day5_tests(path="day5_tests.csv"):

    total = 0
    passed = 0

    with open(path, "r", encoding="utf-8") as file:

        reader = csv.DictReader(file)

        for row in reader:

            total += 1

            question = row["question"]
            expected = row["must_include"].lower()

            # Fresh thread for every independent test
            answer = chat(
                f"test_thread_{total}",
                question
            )

            success = expected in answer.lower()

            if success:
                passed += 1
                status = "PASS"
            else:
                status = "FAIL"

            print(f"\nTest {total}: {status}")
            print("Question:", question)
            print("Answer:", answer)

    rate = (passed / total) * 100 if total else 0

    print("\n" + "=" * 40)
    print("EDUPREDICTAI DAY 5 TEST REPORT")
    print("=" * 40)
    print("Total tests:", total)
    print("Passed:", passed)
    print("Failed:", total - passed)
    print(f"Pass rate: {rate:.2f}%")

    if rate >= 80:
        print("STATUS: TARGET ACHIEVED!")
    else:
        print("STATUS: Improve the agent and test again.")

    return rate


day5_rate = run_day5_tests()


Test 1: FAIL
Question: What is EduPredictAI?
Answer: EduPredictAI is an AI‑powered tool that helps universities with admissions and academic support. It analyzes student data to spot those who may need extra help and gives insights for better academic decisions.

Test 2: PASS
Question: Which technologies are used for the frontend?
Answer: The frontend is built with React.js, using Vite as the build tool. It’s written in JavaScript and styled with Tailwind CSS.

Test 3: PASS
Question: What is the backend technology?
Answer: The backend is built with Python using FastAPI, and it stores data in a MongoDB database.

Test 4: FAIL
Question: Which machine learning libraries are used?
Answer: The project uses Python with the following ML libraries: **Pandas, NumPy, and Scikit‑learn**. These are used for data handling, numerical operations, and building the predictive models.

Test 5: PASS
Question: What are the performance categories?
Answer: The performance categories are:

- **Pass** – scor

Day- 5 Task


In [18]:

%pip install -q chromadb


In [19]:

import chromadb

from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

print("All Day 5 libraries are ready!")


All Day 5 libraries are ready!


In [20]:

EDUPREDICTAI_FACTS = [
    "EduPredictAI is an AI-powered university admissions and academic decision-support application.",
    "EduPredictAI helps universities review student information and make more informed admission decisions.",
    "EduPredictAI uses historical student data to support academic performance prediction.",
    "Student information can include Intermediate first-year marks and second-year marks.",
    "Student information can include MPC subject marks, overall percentage, entrance examination scores, and previous board details.",
    "EduPredictAI can predict academic performance using machine learning models.",
    "The academic performance categories used in the project's student dataset are Pass, Fail, and Dropout.",
    "In the project's G3-based classification, a score from 10 to 20 is Pass.",
    "In the project's G3-based classification, a score from 1 to 9 is Fail.",
    "In the project's G3-based classification, a score of 0 is Dropout.",
    "EduPredictAI can support academic branch and class placement recommendations.",
    "EduPredictAI can screen students for possible merit-based scholarship eligibility.",
    "EduPredictAI can also screen students for possible financial-need-based scholarship eligibility.",
    "Scholarship screening is decision support and does not guarantee that a student will receive a scholarship.",
    "EduPredictAI can help identify students who may need additional academic support.",
    "The application is designed for university administrators, admission staff, and academic counselors.",
    "The planned technology stack includes React, Vite, JavaScript, Tailwind CSS, Python FastAPI, MongoDB, Pandas, NumPy, and Scikit-learn.",
    "EduPredictAI is intended to present insights through dashboards, analytics, and downloadable reports."
]

# Create or reuse the ChromaDB collection
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="edupredictai_knowledge"
)

# Store one fact in each document/chunk
fact_ids = [f"edupredictai_fact_{i+1}" for i in range(len(EDUPREDICTAI_FACTS))]

collection.upsert(
    ids=fact_ids,
    documents=EDUPREDICTAI_FACTS,
    metadatas=[
        {"source": "EduPredictAI project knowledge", "fact_number": i+1}
        for i in range(len(EDUPREDICTAI_FACTS))
    ]
)

print("Knowledge base created successfully!")
print("Number of facts:", collection.count())


Knowledge base created successfully!
Number of facts: 36


In [21]:

@tool
def search_edupredictai_knowledge(question: str) -> str:
    """
    Search the EduPredictAI knowledge base.

    Topics covered:
    - Project purpose and features
    - Student information and academic records
    - Academic performance prediction
    - Pass, Fail, and Dropout classification
    - Branch and class placement recommendations
    - Merit-based and financial-need-based scholarship screening
    - Student academic support
    - User roles
    - Technology stack
    - Dashboards, analytics, and reports

    Use this tool when answering questions about EduPredictAI.
    Do not invent information that is not found in the retrieved facts.
    """
    results = collection.query(
        query_texts=[question],
        n_results=4
    )

    documents = results.get("documents", [[]])[0]

    if not documents:
        return "No matching information was found in the EduPredictAI knowledge base."

    return "\n".join(
        f"- {document}" for document in documents
    )


# Test the search tool
search_result = search_edupredictai_knowledge.invoke(
    "What performance categories does EduPredictAI use?"
)

print(search_result)


- EduPredictAI is intended to present insights through dashboards, analytics, and downloadable reports.
- EduPredictAI is an AI-powered university admission and academic decision-support application.
- EduPredictAI is an AI-powered university admissions and academic decision-support application.
- EduPredictAI provides interactive analytics to help understand student academic performance.


In [22]:

# Reuse Day 4 tools if they are available in this notebook
day4_tool_names = [
    "calculate_percentage",
    "classify_performance",
    "log_student_record"
]

day5_tools = [search_edupredictai_knowledge]

for tool_name in day4_tool_names:
    existing_tool = globals().get(tool_name)

    if existing_tool is not None and callable(existing_tool):
        day5_tools.append(existing_tool)
        print("Added Day 4 tool:", tool_name)
    else:
        print("Day 4 tool not currently loaded:", tool_name)

# Checkpointer stores conversation history separately for each thread_id
checkpointer = InMemorySaver()

SYSTEM_RULES = """
You are EduPredictAI Assistant, an assistant for an AI-powered university
admissions and academic decision-support application.

RULES:
1. Use search_edupredictai_knowledge for questions about EduPredictAI.
2. Answer using the retrieved knowledge and the user's current conversation.
3. Never invent scholarship amounts, eligibility cutoffs, admission rules,
   guarantees, fees, or other facts that are not documented.
4. If the knowledge base does not contain the requested information,
   clearly say: "I don't have that exact information in my knowledge base.
   I will need to check with the university admissions team."
5. Do not pretend that you have contacted the university or checked an
   external system.
6. Explain academic predictions and scholarship screening as decision
   support, not guaranteed outcomes.
7. Keep answers clear, helpful, and easy to understand.
8. Keep each thread's conversation history separate.
"""

agent = create_agent(
    model=llm,
    tools=day5_tools,
    system_prompt=SYSTEM_RULES,
    checkpointer=checkpointer
)

def chat(question, thread_id="edupredictai_default"):
    response = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": thread_id}}
    )

    return response["messages"][-1].content

print("\nEduPredictAI agent created successfully!")
print("Tools available:", [t.name for t in day5_tools])


Day 4 tool not currently loaded: calculate_percentage
Day 4 tool not currently loaded: classify_performance
Day 4 tool not currently loaded: log_student_record

EduPredictAI agent created successfully!
Tools available: ['search_edupredictai_knowledge']


In [23]:

# Thread 1: give the agent a demo student ID
answer1 = chat(
    "For this test, remember the demo student ID EDU101.",
    thread_id="student_thread_1"
)

print("Thread 1:", answer1)

# Ask the same thread to recall the ID
answer2 = chat(
    "What demo student ID did I ask you to remember?",
    thread_id="student_thread_1"
)

print("\nThread 1 follow-up:", answer2)

# Thread 2: this is a separate conversation
answer3 = chat(
    "What demo student ID did I ask you to remember?",
    thread_id="student_thread_2"
)

print("\nThread 2:", answer3)


Thread 1: Got it! I’ll keep the demo student ID **EDU101** in mind for this test. Let me know what you’d like to do next.

Thread 1 follow-up: You asked me to remember the demo student ID **EDU101**.

Thread 2: I don’t have that exact information in my knowledge base. I will need to check with the university admissions team.


In [24]:

unknown_answer = chat(
    "What exact scholarship amount will a student receive, "
    "and what is the minimum family income required to qualify?",
    thread_id="unknown_test"
)

print(unknown_answer)


I don’t have that exact information in my knowledge base. I will need to check with the university admissions team.


In [25]:

# Get fresh agent answers for the functional tests
purpose_answer = chat(
    "What is the main purpose of EduPredictAI?",
    thread_id="test_purpose"
)

classification_answer = chat(
    "Explain the G3 Pass, Fail, and Dropout classification rules.",
    thread_id="test_classification"
)

scholarship_answer = chat(
    "What types of scholarship eligibility can EduPredictAI screen for?",
    thread_id="test_scholarship"
)

# Convert answers to lowercase for easier checking
purpose_text = str(purpose_answer).lower()
classification_text = str(classification_answer).lower()
scholarship_text = str(scholarship_answer).lower()
memory_text = str(answer2).lower()
isolation_text = str(answer3).lower()
unknown_text = str(unknown_answer).lower()
retrieval_text = str(search_result).lower()

tests = [
    (
        "Knowledge base contains 18 facts",
        collection.count() == 18
    ),
    (
        "Search tool retrieves classification facts",
        all(word in retrieval_text for word in ["pass", "fail", "dropout"])
    ),
    (
        "Agent explains the project purpose",
        "admission" in purpose_text or "university" in purpose_text
    ),
    (
        "Agent explains G3 classification",
        all(word in classification_text for word in ["pass", "fail", "dropout"])
        and "10" in classification_text
    ),
    (
        "Agent explains scholarship screening",
        "scholarship" in scholarship_text
        and (
            "merit" in scholarship_text
            or "financial" in scholarship_text
            or "need" in scholarship_text
        )
    ),
    (
        "Agent remembers information in the same thread",
        "edu101" in memory_text
    ),
    (
        "Agent keeps different threads separate",
        "edu101" not in isolation_text
    ),
    (
        "Agent handles unknown information safely",
        any(phrase in unknown_text for phrase in [
            "don't have",
            "do not have",
            "not available",
            "not specified",
            "need to check",
            "check with"
        ])
    )
]

passed = 0

print("EDUPREDICTAI DAY 5 TEST REPORT")
print("-" * 45)

for test_name, result in tests:
    status = "PASS" if result else "FAIL"

    print(f"{status}: {test_name}")

    if result:
        passed += 1

total = len(tests)
pass_rate = (passed / total) * 100

print("-" * 45)
print(f"Passed: {passed}/{total}")
print(f"Failed: {total - passed}/{total}")
print(f"Pass rate: {pass_rate:.2f}%")

if pass_rate >= 80:
    print("Sign-off: Target achieved!")
else:
    print("Sign-off: Target not achieved. Review failed tests.")


EDUPREDICTAI DAY 5 TEST REPORT
---------------------------------------------
FAIL: Knowledge base contains 18 facts
FAIL: Search tool retrieves classification facts
PASS: Agent explains the project purpose
PASS: Agent explains G3 classification
PASS: Agent explains scholarship screening
PASS: Agent remembers information in the same thread
PASS: Agent keeps different threads separate
PASS: Agent handles unknown information safely
---------------------------------------------
Passed: 6/8
Failed: 2/8
Pass rate: 75.00%
Sign-off: Target not achieved. Review failed tests.
